# Lab 12 — El borrador del diagnóstico

**Alumno:** Juan Diego Hernandez Zavala  
**Matrícula:** 2209978x  
**Entidad asignada:** Baja California  

> Resuelve las celdas marcadas con `TODO`. Cuando termines, descarga el notebook (`Archivo → Descargar → .ipynb`) y súbelo en la página de la actividad. No borres esta celda.


# Lab — Sesión 12: El borrador del diagnóstico

**Simulación avanzada e implementación de modelos poblacionales y de catástrofe (ACD53)**

Taller. Cuatro partes, y **ninguna pide un cálculo nuevo**: todo sale de tus labs 1 a 11.
Se levanta el inventario de piezas con su estado, se corren las verificaciones cruzadas
entre labs con semáforo, se escriben las **cuatro frases del diagnóstico** con tus
números, y se arma el plan para la Entrega 1 del jueves (sesión 13): huecos priorizados,
guion de tres minutos y la pregunta que te queda abierta.

Ten abiertos tus notebooks de los labs 1 a 11: de ahí se **copian** los números.


In [194]:
import numpy as np
import pandas as pd

ENTIDAD = "Baja California"   # <- tu entidad
FUENTE  = "Labs 1-2 y 11 (censos INEGI 1990 y 2020), labs 3-4 y 6 (tabla de vida con defunciones INEGI 2020), lab 5 (estandarizacion, datos de practica), labs 7-8 y 10 (fecundidad y TNR), lab 9 (EIC 2015, Censo 2020, nacimientos y defunciones registrados 2015-2019)"   # <- resumen: de qué labs vienen tus números


## Datos

No hay datos nuevos que bajar. Lo que hace falta está en tus labs:

| pieza | lab | fuente oficial |
|---|---|---|
| Población por sexo y edad, censo 1990 | 1-2, 11 | INEGI, Censo 1990 |
| Población por sexo y edad, censo 2020 | 1-2, 11 | INEGI, Censo 2020 |
| Población 2015 (o 2010) | 9 | INEGI, Encuesta Intercensal 2015 |
| Defunciones por edad, 2020 | 3-5 | INEGI, Estadísticas de mortalidad |
| Defunciones del periodo intercensal | 9 | INEGI, Estadísticas de mortalidad |
| Nacimientos por edad de la madre, 2020 | 7-8 | INEGI, Estadísticas de natalidad |
| Nacimientos del periodo intercensal | 9 | INEGI, Estadísticas de natalidad |
| Tabla de vida de la entidad | 4, 6 | tu lab 4 |
| Tabla de vida nacional | 4, 6 | CONAPO / tu lab 4 |
| Tasas nacionales por edad (estándar) | 5 | INEGI / CONAPO |
| Indicadores de estructura 1990 y 2020 | 11 | tu lab 11 |

Los datos de práctica son los de la entidad ficticia de los labs 6 a 9, reunidos.


In [195]:
# --- datos de PRACTICA (entidad ficticia de los labs 6-9; no son de ninguna entidad) ---
PRACTICA = dict(
    e0_lab4 = 74.334,   e0_lab6 = 74.3334,   e0_pais = 75.551,
    brecha_e0 = 1.2171, suma_arriaga = 1.2171,
    TBM = 5.416,        suma_cM = 5.416,     TBM_estand = 9.163, TBM_pais = 8.434,
    TGF = 2.0975,       TBR = 1.0232,        TNR = 0.9844, umbral_reemplazo = 2.1308,
    P2015 = 1_860_000,  P2020 = 2_110_000,   CN = 137_000, SM = 113_000,
    TCN = 13.804,       TMN = 11.385,        r = 25.189,
    SM_rango = (113_000, 155_000),
    men15_2020 = 25.5,  may65_2020 = 6.0,    dependencia_2020 = 0.459, mediana_2020 = 29.1,
    men15_1990 = 42.5,  dependencia_1990 = 0.842, mediana_1990 = 18.3,
)

PIEZAS = ["censo 1990 por edad y sexo", "censo 2020 por edad y sexo", "poblacion 2015",
          "defunciones 2020 por edad", "defunciones del periodo", "nacimientos 2020 por edad de la madre",
          "nacimientos del periodo", "tabla de vida de la entidad", "tabla de vida nacional",
          "tasas nacionales por edad (estandar)", "indicadores de estructura 1990 y 2020"]
LABS   = ["1-2, 11", "1-2, 11", "9", "3-5", "9", "7-8", "9", "4, 6", "4, 6", "5", "11"]


def semaforo(nombre, ok, detalle=""):
    """Imprime una verificacion con su resultado. El rojo no es un error tuyo:
    es la senal de que dos labs no dicen lo mismo y hay que averiguar cual."""
    print(f"{'OK     ' if ok else 'REVISAR'} | {nombre} {detalle}")
    return ok


def costo_practica(tabla):
    """Cuanto cuesta en la rubrica de la Entrega 1 entregar con lo que hay:
    0.5 puntos por cada PARTE de la entrega que dependa de una pieza no propia."""
    partes = {"P1 estructura": ["censo 1990 por edad y sexo", "censo 2020 por edad y sexo", "indicadores de estructura 1990 y 2020"],
              "P2 mortalidad": ["defunciones 2020 por edad", "tasas nacionales por edad (estandar)"],
              "P3 tabla de vida": ["tabla de vida de la entidad", "tabla de vida nacional"],
              "P4 fecundidad y crecimiento": ["nacimientos 2020 por edad de la madre", "poblacion 2015",
                                             "nacimientos del periodo", "defunciones del periodo"]}
    estado = dict(zip(tabla["pieza"], tabla["estado"]))
    perdida = {p: (0.0 if all(estado.get(x) == "PROPIO" for x in xs) else 0.5) for p, xs in partes.items()}
    return perdida, sum(perdida.values())


## Tus datos

In [196]:
# TODO 0: escribe ENTIDAD y FUENTE arriba, y pega aqui TUS numeros copiados de los labs
#   (mismas llaves que PRACTICA). Deja None en lo que aun no tengas: esa pieza corre con
#   la practica Y LO IMPRIME, que es justo lo que la parte 1 va a inventariar.
MIS = dict(
    e0_lab4 = 74.7430, e0_lab6 = 74.7430, e0_pais = 76.08,        # lab 4 (Tx/lx) y lab 6; pais = referencia del lab 4
    brecha_e0 = 0.8914, suma_arriaga = 0.8914,                     # lab 6 (contra Michoacan, no contra el pais)
    TBM = None, suma_cM = None, TBM_estand = None, TBM_pais = None,   # lab 5 se hizo con la entidad de PRACTICA
    TGF = 1.8225, TBR = 0.8890, TNR = 0.8745, umbral_reemplazo = 2.0840,   # lab 8
    P2015 = 3_315_766, P2020 = 3_769_020, CN = 209_078, SM = 244_176,     # lab 9
    TCN = 11.8043, TMN = 13.7859, r = 25.5903,                            # lab 9, por mil
    SM_rango = (234_744, 321_095),                                        # lab 9, parte 3
    men15_2020 = 23.35, may65_2020 = 6.52, dependencia_2020 = 0.426, mediana_2020 = 30.3,   # censo 2020 (labs 1 y 11)
    men15_1990 = 34.91, dependencia_1990 = 0.623, mediana_1990 = 21.2,                      # censo 1990 (labs 1 y 11)
)

D = {}
for k, v in PRACTICA.items():
    propio = MIS.get(k) is not None
    D[k] = MIS[k] if propio else v
    print(f"{k:18s} {'PROPIO' if propio else 'PRACTICA'}")


e0_lab4            PROPIO
e0_lab6            PROPIO
e0_pais            PROPIO
brecha_e0          PROPIO
suma_arriaga       PROPIO
TBM                PRACTICA
suma_cM            PRACTICA
TBM_estand         PRACTICA
TBM_pais           PRACTICA
TGF                PROPIO
TBR                PROPIO
TNR                PROPIO
umbral_reemplazo   PROPIO
P2015              PROPIO
P2020              PROPIO
CN                 PROPIO
SM                 PROPIO
TCN                PROPIO
TMN                PROPIO
r                  PROPIO
SM_rango           PROPIO
men15_2020         PROPIO
may65_2020         PROPIO
dependencia_2020   PROPIO
mediana_2020       PROPIO
men15_1990         PROPIO
dependencia_1990   PROPIO
mediana_1990       PROPIO


## Parte 1 — El inventario  *(2.5 puntos)*

El error más caro medido en los labs 1 a 9 es entregar con datos de práctica (lab01 P5:
75% de las entregas; lab02 P2: 62%). En la Entrega 1 cuesta **0.5 de cada 2.5** en la
parte que dependa de esa pieza. Aquí se ve, pieza por pieza, qué tienes.


In [197]:
# TODO 1.1: llena la tabla de inventario: para cada pieza de PIEZAS, su lab (LABS), la
#   FUENTE oficial concreta (institucion, nombre de la tabla, anio) y el ESTADO: "PROPIO",
#   "PRACTICA" o "FALTA". Revisa el TODO 0 de cada lab: si imprimio «Usando los datos de
#   PRACTICA», esa pieza es PRACTICA aunque creas que ya la tenias. Imprime la tabla completa.
FUENTES = [
    "INEGI, XI Censo General de Población y Vivienda 1990: 'Poblacion total por entidad federativa y grupo quinquenal de edad según sexo, serie 1990-2020'",
    "INEGI, Censo de Población y Vivienda 2020: mismo tabulado, serie 1990-2020",
    "INEGI, Encuesta Intercensal 2015: población total por entidad (3,315,766)",
    "INEGI, Estadísticas de Defunciones Registradas 2020, consulta por grupo de edad (lab 2, INEGI_exporta)",
    "INEGI, tabulado 'Defunciones registradas por entidad de residencia habitual según sexo, 2010-2025' (2015-2019)",
    "INEGI/CONAPO, estimaciones 2023 basadas en la ENR (lab 7; cifras redondeadas, verificar)",
    "INEGI, tabulado 'Nacimientos registrados por entidad de residencia habitual de la madre según sexo, 2010-2025' (2015-2019)",
    "Propia (labs 3-4): nMx con defunciones INEGI 2020 y población del Censo 2020",
    "Tabla de referencia ilustrativa del curso (labs 3-4); lab 6 contra Michoacán con tasas ilustrativas",
    "Referencia del curso (lab 5); el lab 5 se resolvió con la entidad de PRACTICA",
    "Propios: calculados de los censos 1990 y 2020 de INEGI (labs 1 y 11)",
]
ESTADOS = ["PROPIO", "PROPIO", "PROPIO", "PROPIO", "PROPIO", "PROPIO", "PROPIO", "PROPIO",
           "PRACTICA", "PRACTICA", "PROPIO"]
inventario = pd.DataFrame({"pieza": PIEZAS, "lab": LABS,
                           "fuente": FUENTES,
                           "estado": ESTADOS})
pd.set_option('display.max_colwidth', None)
display(inventario)

# TODO 1.2: cuenta cuantas piezas y que porcentaje estan en PROPIO, y usa costo_practica()
#   para imprimir cuanto perderias en la rubrica de la Entrega 1 si entregaras HOY asi.
n_propio = (inventario["estado"] == "PROPIO").sum()
print(f"\nPiezas PROPIAS: {n_propio} de {len(inventario)} ({n_propio / len(inventario) * 100:.0f}%)")
perdida, total_perdida = costo_practica(inventario)
for parte, p in perdida.items():
    print(f"  {parte:30s} pierde {p:.1f}")
print(f"Si entregara hoy así, perdería {total_perdida:.1f} puntos de la Entrega 1 (P2 mortalidad y P3 tabla de vida).")

# TODO 1.3: para cada pieza en PRACTICA o FALTA, escribe de donde la vas a sacar —tabla
#   oficial concreta— y cuanto tiempo estimas. Imprime esa lista. «La busco» no cuenta.
PLAN = {
    "tabla de vida nacional":
        ("INEGI, Estadisticas de Defunciones Registradas 2020 (total nacional por grupo de edad) entre la población "
         "nacional del Censo 2020 por edad; contraste con la e0 2020 de CONAPO (Conciliación Demográfica). "
         "Con eso rehago el Arriaga del lab 6 contra el pais", "1.5 h"),
    "tasas nacionales por edad (estandar)":
        ("Las mismas nMx nacionales 2020 y la estructura nacional del Censo 2020 como estandar; rehacer el lab 5 "
         "con la estructura y las nMx de Baja California de los labs 3-4", "1.5 h"),
}
for pieza, (fuente, tiempo) in PLAN.items():
    print(f"- {pieza}: {fuente}. Tiempo estimado: {tiempo}.")

,pieza,lab,fuente,estado
0,censo 1990 por edad y sexo,"1-2, 11","INEGI, XI Censo General de Población y Vivienda 1990: 'Poblacion total por entidad federativa y grupo quinquenal de edad según sexo, serie 1990-2020'",PROPIO
1,censo 2020 por edad y sexo,"1-2, 11","INEGI, Censo de Población y Vivienda 2020: mismo tabulado, serie 1990-2020",PROPIO
2,poblacion 2015,9,"INEGI, Encuesta Intercensal 2015: población total por entidad (3,315,766)",PROPIO
3,defunciones 2020 por edad,3-5,"INEGI, Estadísticas de Defunciones Registradas 2020, consulta por grupo de edad (lab 2, INEGI_exporta)",PROPIO
4,defunciones del periodo,9,"INEGI, tabulado 'Defunciones registradas por entidad de residencia habitual según sexo, 2010-2025' (2015-2019)",PROPIO
5,nacimientos 2020 por edad de la madre,7-8,"INEGI/CONAPO, estimaciones 2023 basadas en la ENR (lab 7; cifras redondeadas, verificar)",PROPIO
6,nacimientos del periodo,9,"INEGI, tabulado 'Nacimientos registrados por entidad de residencia habitual de la madre según sexo, 2010-2025' (2015-2019)",PROPIO
7,tabla de vida de la entidad,"4, 6",Propia (labs 3-4): nMx con defunciones INEGI 2020 y población del Censo 2020,PROPIO
8,tabla de vida nacional,"4, 6",Tabla de referencia ilustrativa del curso (labs 3-4); lab 6 contra Michoacán con tasas ilustrativas,PRACTICA
9,tasas nacionales por edad (estandar),5,Referencia del curso (lab 5); el lab 5 se resolvió con la entidad de PRACTICA,PRACTICA



Piezas PROPIAS: 9 de 11 (82%)
  P1 estructura                  pierde 0.0
  P2 mortalidad                  pierde 0.5
  P3 tabla de vida               pierde 0.5
  P4 fecundidad y crecimiento    pierde 0.0
Si entregara hoy así, perdería 1.0 puntos de la Entrega 1 (P2 mortalidad y P3 tabla de vida).
- tabla de vida nacional: INEGI, Estadisticas de Defunciones Registradas 2020 (total nacional por grupo de edad) entre la población nacional del Censo 2020 por edad; contraste con la e0 2020 de CONAPO (Conciliación Demográfica). Con eso rehago el Arriaga del lab 6 contra el pais. Tiempo estimado: 1.5 h.
- tasas nacionales por edad (estandar): Las mismas nMx nacionales 2020 y la estructura nacional del Censo 2020 como estandar; rehacer el lab 5 con la estructura y las nMx de Baja California de los labs 3-4. Tiempo estimado: 1.5 h.


## Parte 2 — Las verificaciones cruzadas  *(2.5 puntos)*

El segundo error más caro: no cruzar entregas (lab06 P1.2, 50%). Cinco verificaciones
que no cuestan nada y que la Entrega 1 exige impresas. Un semáforo en rojo **no es un
error del lab**: es la señal de que dos labs no dicen lo mismo, y hay que averiguar cuál.


In [198]:
# TODO 2.1: compara la e0 de tu lab 4 con la del lab 6 con tolerancia de 0.05 anios y
#   pasa el resultado por semaforo(). Si difieren en milesimas es el redondeo de los nax y
#   esta bien; si difieren en decimas, usaron tablas distintas: di EN TEXTO cual dato cambio.
ok_e0 = semaforo("e0 lab 4 vs lab 6", abs(D["e0_lab4"] - D["e0_lab6"]) < 0.05,
                 f"({D['e0_lab4']:.4f} vs {D['e0_lab6']:.4f}, diferencia {abs(D['e0_lab4'] - D['e0_lab6']):.4f})")
print("Coinciden: el lab 6 usó las mismas diez nMx de Baja California del lab 4.")

# TODO 2.2: verifica EN CODIGO e imprime con semaforo() que TBM == suma de c·M (sesion 5) y
#   que la suma de Arriaga reproduce la brecha de e0 con el pais (sesion 6). Tolerancia 0.005.
ok_tbm = semaforo("TBM = suma c*M (lab 5)", abs(D["TBM"] - D["suma_cM"]) < 0.005,
                  f"({D['TBM']:.3f} vs {D['suma_cM']:.3f}) -- con datos de PRACTICA")
ok_arr = semaforo("Arriaga reproduce la brecha (lab 6)", abs(D["brecha_e0"] - D["suma_arriaga"]) < 0.005,
                  f"({D['brecha_e0']:.4f} vs {D['suma_arriaga']:.4f})")
print("Aviso: la brecha del lab 6 es contra Michoacán, no contra el país. La brecha con la referencia nacional "
      f"del lab 4 es {D['e0_pais'] - D['e0_lab4']:.2f} años y todavía no esta descompuesta por Arriaga.")

# TODO 2.3: verifica EN CODIGO e imprime con semaforo() que CN + SM reproduce el cambio de
#   poblacion 2015-2020 y que TCN + TMN da r (lab 9), y que la TNR se compara con TU umbral
#   de reemplazo (TGF/(p·frac) del lab 8) y no con 2.1: imprime TGF, umbral y el veredicto.
dP = D["P2020"] - D["P2015"]
ok_cn = semaforo("CN + SM = cambio 2015-2020", D["CN"] + D["SM"] == dP,
                 f"({D['CN']:,} + {D['SM']:,} = {D['CN'] + D['SM']:,} vs {dP:,})")
ok_r = semaforo("TCN + TMN = r", abs(D["TCN"] + D["TMN"] - D["r"]) < 0.005,
                f"({D['TCN']:.4f} + {D['TMN']:.4f} = {D['TCN'] + D['TMN']:.4f} vs {D['r']:.4f})")
print(f"\nTGF = {D['TGF']:.4f} | mi umbral de reemplazo = {D['umbral_reemplazo']:.4f} (no 2.1) | TNR = {D['TNR']:.4f}")
veredicto = "sí se reemplaza" if D["TNR"] >= 1 else "no se reemplaza"
ok_rep = semaforo("TNR contra 1 y TGF contra mí umbral",
                  (D["TNR"] < 1) == (D["TGF"] < D["umbral_reemplazo"]),
                  f"-> {ENTIDAD} {veredicto}: TGF {D['TGF']:.4f} < umbral {D['umbral_reemplazo']:.4f} y TNR {D['TNR']:.4f} < 1")

OK      | e0 lab 4 vs lab 6 (74.7430 vs 74.7430, diferencia 0.0000)
Coinciden: el lab 6 usó las mismas diez nMx de Baja California del lab 4.
OK      | TBM = suma c*M (lab 5) (5.416 vs 5.416) -- con datos de PRACTICA
OK      | Arriaga reproduce la brecha (lab 6) (0.8914 vs 0.8914)
Aviso: la brecha del lab 6 es contra Michoacán, no contra el país. La brecha con la referencia nacional del lab 4 es 1.34 años y todavía no esta descompuesta por Arriaga.
OK      | CN + SM = cambio 2015-2020 (209,078 + 244,176 = 453,254 vs 453,254)
OK      | TCN + TMN = r (11.8043 + 13.7859 = 25.5902 vs 25.5903)

TGF = 1.8225 | mi umbral de reemplazo = 2.0840 (no 2.1) | TNR = 0.8745
OK      | TNR contra 1 y TGF contra mí umbral -> Baja California no se reemplaza: TGF 1.8225 < umbral 2.0840 y TNR 0.8745 < 1


## Parte 3 — Las cuatro frases  *(2.5 puntos)*

Una frase por componente, con cuatro piezas nombradas: **afirmación**, **número propio**,
**comparación**, **límite**. Es lo que la rúbrica de la Entrega 1 califica en cada parte.
Los números van impresos desde variables (`D[...]`), no escritos a mano.


In [199]:
# TODO 3.1 — ESTRUCTURA. Escribe la frase de la piramide y la memoria del sistema
#   (sesion 11): que le paso a la base, al centro y a la cuspide entre 1990 y 2020, con la
#   razon de dependencia y la mediana como numero, la comparacion (el otro censo, el pais)
#   y el limite (que no dice la piramide sola). Imprimela con f-string desde D.
print(f"ESTRUCTURA. Entre 1990 y 2020, la pirámide de {ENTIDAD} se estrechó en la base (menores de 15: "
      f"{D['men15_1990']:.1f}% -> {D['men15_2020']:.1f}%), se ensanchó en el centro y empezó a crecer en la cúspide "
      f"(65 y mas: {D['may65_2020']:.1f}% en 2020); la razón de dependencia bajo de {D['dependencia_1990']:.3f} a "
      f"{D['dependencia_2020']:.3f} y la edad mediana subió de {D['mediana_1990']:.1f} a {D['mediana_2020']:.1f} años. "
      "Frente a su población estable (lab 10) sigue siendo muy joven, y el abultamiento de 20-24 (cohorte 1996-2000) "
      "es memoria de inmigración reciente, no de nacimientos. Límite: la pirámide sola no separa cuánto de esa forma "
      "es fecundidad, cuánto migración y cuánto error censal (el 0-4 quedó 12% por debajo de sus nacimientos).")

ESTRUCTURA. Entre 1990 y 2020, la pirámide de Baja California se estrechó en la base (menores de 15: 34.9% -> 23.4%), se ensanchó en el centro y empezó a crecer en la cúspide (65 y mas: 6.5% en 2020); la razón de dependencia bajo de 0.623 a 0.426 y la edad mediana subió de 21.2 a 30.3 años. Frente a su población estable (lab 10) sigue siendo muy joven, y el abultamiento de 20-24 (cohorte 1996-2000) es memoria de inmigración reciente, no de nacimientos. Límite: la pirámide sola no separa cuánto de esa forma es fecundidad, cuánto migración y cuánto error censal (el 0-4 quedó 12% por debajo de sus nacimientos).


In [200]:
# TODO 3.2 — MORTALIDAD. La frase de bruta contra estandarizada (sesion 5) y de e0 con
#   el origen de la brecha (sesion 6). Afirmacion, numero, comparacion con el pais, limite
#   (la estandarizada depende del estandar; e0 no es la edad de muerte).
print(f"MORTALIDAD. Baja California vive menos que la referencia nacional: su e0 es {D['e0_lab4']:.2f} años, "
      f"{D['e0_pais'] - D['e0_lab4']:.2f} menos que la referencia nacional ({D['e0_pais']:.2f}), y contra Michoacán la "
      f"brecha de {D['brecha_e0']:.2f} años se concentra en 65-74 (38%) y en 15-34, mientras que en 0-4 la entidad "
      "está mejor. La comparación bruta contra estandarizada aún no es propia: el lab 5 se hizo con datos de PRACTICA "
      f"(TBM {D['TBM']:.2f} vs estandarizada {D['TBM_estand']:.2f} por mil); con datos propios, la TBM de 2020 fue "
      "6.10 por mil (lab 2), baja por una estructura joven. Límite: la estandarizada depende del estándar elegido, "
      "y la e0 no es la edad a la que muere la gente, sino un resumen del régimen de un año.")

MORTALIDAD. Baja California vive menos que la referencia nacional: su e0 es 74.74 años, 1.34 menos que la referencia nacional (76.08), y contra Michoacán la brecha de 0.89 años se concentra en 65-74 (38%) y en 15-34, mientras que en 0-4 la entidad está mejor. La comparación bruta contra estandarizada aún no es propia: el lab 5 se hizo con datos de PRACTICA (TBM 5.42 vs estandarizada 9.16 por mil); con datos propios, la TBM de 2020 fue 6.10 por mil (lab 2), baja por una estructura joven. Límite: la estandarizada depende del estándar elegido, y la e0 no es la edad a la que muere la gente, sino un resumen del régimen de un año.


In [201]:
# TODO 3.3 — REPRODUCCION. La frase de TGF y TNR contra TU umbral de reemplazo (sesion 8),
#   con el limite: la TNR no incluye migracion.
print(f"REPRODUCCIÓN. Baja California no se reemplaza: su TGF de {D['TGF']:.2f} hijos por mujer está "
      f"{D['umbral_reemplazo'] - D['TGF']:.2f} por debajo de su umbral de reemplazo ({D['umbral_reemplazo']:.3f}, no 2.1) "
      f"y su TNR es {D['TNR']:.3f}: cada generación de mujeres es {(1 - D['TNR']) * 100:.1f}% más chica que la anterior "
      "(contra una TNR nacional de 1.003). La diferencia viene de la fecundidad, no de la supervivencia. Límite: la "
      "TNR describe una población cerrada y no incluye la migración, que en esta entidad es la mitad del crecimiento.")

REPRODUCCIÓN. Baja California no se reemplaza: su TGF de 1.82 hijos por mujer está 0.26 por debajo de su umbral de reemplazo (2.084, no 2.1) y su TNR es 0.875: cada generación de mujeres es 12.5% más chica que la anterior (contra una TNR nacional de 1.003). La diferencia viene de la fecundidad, no de la supervivencia. Límite: la TNR describe una población cerrada y no incluye la migración, que en esta entidad es la mitad del crecimiento.


In [202]:
# TODO 3.4 — CRECIMIENTO. La frase de las tasas (TCN, TMN, r), el caso de tu entidad
#   (crece/decrece, por cual via) y el saldo NETO como RANGO (sesion 9), con su limite: el
#   saldo no dice cuanta gente se fue.
print(f"CRECIMIENTO. Baja California crece por las dos vías: {D['r']:.1f} por mil al año, de los cuales "
      f"{D['TCN']:.1f} son crecimiento natural y {D['TMN']:.1f} migración neta, más que lo natural. Su saldo "
      f"migratorio neto 2015-2020 esta entre {D['SM_rango'][0]:,} y {D['SM_rango'][1]:,} personas (central "
      f"{D['SM']:,}, {D['SM'] / D['P2020'] * 100:.1f}% de la población de 2020), aunque su tasa intrínseca es "
      "negativa (-4.94 por mil, lab 10). Límite: el saldo es entradas menos salidas y no dice cuánta gente llegó ni "
      "cuánta se fue; además, su tamaño depende de la cobertura del censo.")

CRECIMIENTO. Baja California crece por las dos vías: 25.6 por mil al año, de los cuales 11.8 son crecimiento natural y 13.8 migración neta, más que lo natural. Su saldo migratorio neto 2015-2020 esta entre 234,744 y 321,095 personas (central 244,176, 6.5% de la población de 2020), aunque su tasa intrínseca es negativa (-4.94 por mil, lab 10). Límite: el saldo es entradas menos salidas y no dice cuánta gente llegó ni cuánta se fue; además, su tamaño depende de la cobertura del censo.


## Parte 4 — El plan para el jueves  *(2.5 puntos)*

La Entrega 1 es el jueves 8 de octubre (sesión 13): notebook completo y presentación de
**tres minutos** con el notebook abierto. Aquí se decide en qué invertir el tiempo que queda.


In [203]:
# TODO 4.1 — LOS HUECOS. Lista lo que falta: piezas en PRACTICA o FALTA (parte 1),
#   verificaciones en REVISAR (parte 2), frases incompletas (parte 3). Ordenalas por lo que
#   valen en la rubrica ENTRE lo que cuestan en tiempo, y di cuales vas a cerrar antes del
#   jueves y cuales vas a DECLARAR en la entrega (una pieza de practica declarada en una
#   frase no es un error: es un limite).
huecos = pd.DataFrame([
    {"hueco": "Estandarización propia (lab 5): tasas nacionales y estructura de BC", "vale": 0.5, "horas": 1.5,
     "acción": "cerrar"},
    {"hueco": "Tabla de vida nacional propia y Arriaga contra el país (labs 4 y 6)", "vale": 0.5, "horas": 1.5,
     "acción": "cerrar si alcanza; si no, declarar"},
    {"hueco": "Frase de mortalidad sin bruta/estandarizada propia (depende del lab 5)", "vale": 0.25, "horas": 0.2,
     "acción": "se cierra con el lab 5"},
    {"hueco": "Tasas de fecundidad del lab 7 redondeadas (verificar contra ENR 2020)", "vale": 0.25, "horas": 1.0,
     "acción": "declarar como límite"},
])
huecos["puntos por hora"] = huecos["vale"] / huecos["horas"]
display(huecos.sort_values("puntos por hora", ascending=False).reset_index(drop=True))
print("Las seis verificaciones de la parte 2 salen ok, pero la de la TBM pasa con datos de PRACTICA: no cuenta como propia.")
print("Antes de la entrega, cierro el lab 5 (mismo valor que la tabla nacional y arregla la frase de mortalidad). "
      "La tabla nacional la cierro si alcanza el tiempo; si no, la declaro en la frase de mortalidad. "
      "La fecundidad redondeada la declaro como limite.")

,hueco,vale,horas,acción,puntos por hora
0,Frase de mortalidad sin bruta/estandarizada propia (depende del lab 5),0.25,0.2,se cierra con el lab 5,1.250000
1,Estandarización propia (lab 5): tasas nacionales y estructura de BC,0.50,1.5,cerrar,0.333333
2,Tabla de vida nacional propia y Arriaga contra el país (labs 4 y 6),0.50,1.5,"cerrar si alcanza; si no, declarar",0.333333
3,Tasas de fecundidad del lab 7 redondeadas (verificar contra ENR 2020),0.25,1.0,declarar como límite,0.250000


Las seis verificaciones de la parte 2 salen ok, pero la de la TBM pasa con datos de PRACTICA: no cuenta como propia.
Antes de la entrega, cierro el lab 5 (mismo valor que la tabla nacional y arregla la frase de mortalidad). La tabla nacional la cierro si alcanza el tiempo; si no, la declaro en la frase de mortalidad. La fecundidad redondeada la declaro como limite.


In [204]:
# TODO 4.2 — EL GUION DE TRES MINUTOS. Con tiempos: 0:00-0:40 una grafica (las dos
#   piramides) y donde esta tu entidad en la transicion; 0:40-2:20 las cuatro frases;
#   2:20-3:00 la pregunta abierta. Sin explicar metodos ni leer tablas. Imprimelo.
guion = [
    ("0:00-0:40", f"Las dos pirámides 1990 y 2020 de {ENTIDAD}: la base se estrecha (menores de 15 de "
                  f"{D['men15_1990']:.0f}% a {D['men15_2020']:.0f}%) y el centro se ensancha. Esta en la fase avanzada "
                  "de la transición: mortalidad y fecundidad bajas, con una estructura todavía joven por la migración."),
    ("0:40-1:05", f"Estructura: mediana de {D['mediana_1990']:.0f} a {D['mediana_2020']:.0f} años, dependencia de "
                  f"{D['dependencia_1990']:.2f} a {D['dependencia_2020']:.2f}; el abultamiento de 20-24 es migración."),
    ("1:05-1:30", f"Mortalidad: e0 de {D['e0_lab4']:.1f} anios, {D['e0_pais'] - D['e0_lab4']:.1f} menos que el país; "
                  "contra Michoacán la brecha esta en 65-74 y en 15-34."),
    ("1:30-1:55", f"Reproducción: TGF {D['TGF']:.2f} contra mi umbral {D['umbral_reemplazo']:.2f}; TNR "
                  f"{D['TNR']:.3f}, no se reemplaza."),
    ("1:55-2:20", f"Crecimiento: {D['r']:.1f} por mil, más de la mitad por migración; saldo neto entre "
                  f"{D['SM_rango'][0] / 1000:.0f} y {D['SM_rango'][1] / 1000:.0f} mil personas."),
    ("2:20-3:00", "La pregunta abierta (4.3) y el dato que la contestaría."),
]
for t, texto in guion:
    print(f"{t}  {texto}")

0:00-0:40  Las dos pirámides 1990 y 2020 de Baja California: la base se estrecha (menores de 15 de 35% a 23%) y el centro se ensancha. Esta en la fase avanzada de la transición: mortalidad y fecundidad bajas, con una estructura todavía joven por la migración.
0:40-1:05  Estructura: mediana de 21 a 30 años, dependencia de 0.62 a 0.43; el abultamiento de 20-24 es migración.
1:05-1:30  Mortalidad: e0 de 74.7 anios, 1.3 menos que el país; contra Michoacán la brecha esta en 65-74 y en 15-34.
1:30-1:55  Reproducción: TGF 1.82 contra mi umbral 2.08; TNR 0.875, no se reemplaza.
1:55-2:20  Crecimiento: 25.6 por mil, más de la mitad por migración; saldo neto entre 235 y 321 mil personas.
2:20-3:00  La pregunta abierta (4.3) y el dato que la contestaría.


In [205]:
# TODO 4.3 — LA PREGUNTA ABIERTA. UNA pregunta concreta sobre TU entidad: el dato que mas
#   dudas o lo que el diagnostico no explica. Di que dato la contestaria. La sesion 14 la
#   convierte en hipotesis de prediccion.
print(f"Pregunta: ¿la fecundidad de {ENTIDAD} es realmente de {D['TGF']:.2f} hijos por mujer, o es mucho más baja? "
      "Mis tasas del lab 7 son cifras redondeadas, y con la Encuesta Intercensal 2025, se reportó para la entidad una "
      "TGF cercana a 1.0. Si fuera así, la TNR y la tasa intrínseca serían mucho más negativas y el crecimiento "
      "dependería todavía más de la migración.")
print("Dato que la contestaría: las tasas específicas de fecundidad por edad de la madre de la Encuesta Intercensal "
      "2025 de INEGI (o las de CONAPO para 2020-2025), comparadas con los nacimientos ocurridos por edad de la madre "
      "de la Estadística de Nacimientos Registrados, para separar el nivel real del efecto del registro extemporáneo.")

Pregunta: ¿la fecundidad de Baja California es realmente de 1.82 hijos por mujer, o es mucho más baja? Mis tasas del lab 7 son cifras redondeadas, y con la Encuesta Intercensal 2025, se reportó para la entidad una TGF cercana a 1.0. Si fuera así, la TNR y la tasa intrínseca serían mucho más negativas y el crecimiento dependería todavía más de la migración.
Dato que la contestaría: las tasas específicas de fecundidad por edad de la madre de la Encuesta Intercensal 2025 de INEGI (o las de CONAPO para 2020-2025), comparadas con los nacimientos ocurridos por edad de la madre de la Estadística de Nacimientos Registrados, para separar el nivel real del efecto del registro extemporáneo.


## Cierre y entrega

**Checklist:**
- [ ] El inventario tiene las once piezas con **fuente concreta** y estado.
- [ ] Las cinco verificaciones están **impresas** con su semáforo, y cada rojo tiene explicación.
- [ ] Las cuatro frases tienen afirmación, número **desde variable**, comparación y límite.
- [ ] El saldo está como **rango** y se llama **saldo neto**; la TNR se compara con **mi** umbral.
- [ ] Los huecos están priorizados y el guion cabe en tres minutos.

**Cómo se entrega:** en Colab, `Archivo → Descargar → Descargar .ipynb`, y arrastra ese
archivo al sistema. **El archivo, no el enlace.** Ejecuta todo antes de descargar: el
evaluador lee tus salidas, no las genera.
